# Week 64, Day 5: Gaussian processes for neural data and experiments

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w064/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: GPFA on a population

Complete the GPFA posterior: build the block-diagonal prior covariance K over the two latents, the stacked loading matrix, and solve for the posterior mean. Compare with smoothing each neuron and projecting.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import block_diag
g0 = np.random.default_rng(645)
T, N, L = 120, 25, 2
t = np.arange(T)
taus = np.array([25.0, 8.0])
Kl = [np.exp(-0.5 * ((t[:, None] - t[None, :]) / tau) ** 2) + 1e-6 * np.eye(T) for tau in taus]
Ztrue = np.column_stack([np.linalg.cholesky(K_) @ g0.normal(size=T) for K_ in Kl])
C = g0.normal(0, 1.0, (N, L)); dvec = g0.normal(0, 0.3, N)
Psi = g0.uniform(0.5, 2.0, N)
X = Ztrue @ C.T + dvec + g0.normal(size=(T, N)) * np.sqrt(Psi)

K_big = block_diag(*Kl)                                        # (L T) x (L T), latent-major order
C_big = np.zeros((T * N, T * L))
for i in range(T):
    C_big[i * N:(i + 1) * N, :][:, [j * T + i for j in range(L)]] = C
Psi_big = np.tile(Psi, T)
xvec = (X - dvec).reshape(-1)
Prec = ...                                                     # posterior precision
mu = np.linalg.solve(Prec, C_big.T @ (xvec / Psi_big))
Zpost = mu.reshape(L, T).T

def best_match(Z):
    Cm = np.abs(np.corrcoef(np.hstack([Ztrue, Z]).T)[:L, L:])
    return Cm.max(1)
match = best_match(Zpost)
w = np.exp(-0.5 * ((t[:, None] - t[None, :]) / 10) ** 2); w /= w.sum(1, keepdims=True)
Zsmooth = (w @ (X - dvec)) @ np.linalg.pinv(C).T
match_smooth = best_match(Zsmooth)
print(f"GPFA |corr| with the true latents {np.round(match, 3)}; smooth-then-project {np.round(match_smooth, 3)}")
plt.plot(Ztrue[:, 0], "k", label="true latent 1"); plt.plot(Zpost[:, 0], label="GPFA"); plt.plot(Zsmooth[:, 0], alpha=0.7, label="smoothed"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_shapes():
    "the posterior mean has one row per time bin and one column per latent"
    assert Zpost.shape == (T, 2)
def test_recovery():
    "both latents are recovered (absolute correlation above 0.9 after matching)"
    assert min(match) > 0.9
def test_better_than_smoothing():
    "GPFA beats smoothing each neuron then projecting with the true loadings"
    assert np.mean(match) > np.mean(match_smooth)

test_shapes()
test_recovery()
test_better_than_smoothing()
print("All checks passed")

### Exercise: Learning each latent's timescale

Write neg_lml(log_taus), the negative log marginal likelihood of the data with the loadings and noise known, and minimise it over the two log timescales.

In [ ]:
from scipy.optimize import minimize
from scipy.linalg import cho_factor, cho_solve

def neg_lml(log_taus):
    Kb = block_diag(*[np.exp(-0.5 * ((t[:, None] - t[None, :]) / np.exp(lt)) ** 2) + 1e-6 * np.eye(T) for lt in log_taus])
    Cov = C_big @ Kb @ C_big.T + np.diag(Psi_big)
    c, low = cho_factor(Cov, lower=True)
    return ...

res = minimize(neg_lml, np.log([12.0, 12.0]), method="Nelder-Mead", options={"xatol": 0.05, "fatol": 0.5})
taus_hat = np.sort(np.exp(res.x))[::-1]
print(f"learned timescales {np.round(taus_hat, 2)} bins (true {taus})")

In [ ]:
import numpy as np
def test_taus():
    "the learned timescales are within 40% of the true 25 and 8 bins"
    assert abs(taus_hat[0] / 25 - 1) < 0.4 and abs(taus_hat[1] / 8 - 1) < 0.4

test_taus()
print("All checks passed")

### Exercise: Finding the preferred stimulus

Complete the expected improvement and the loop: on each trial fit a GP to the responses so far (length-scale 0.06, signal variance 0.3, noise variance 0.01) and test the stimulus with the largest EI. Compare with random selection, averaged over 20 simulated neurons.

In [ ]:
from scipy.stats import norm
grid = np.linspace(0, 1, 200)
gb = np.random.default_rng(6451)

def response(x, pref):
    return np.exp(-0.5 * ((x - pref) / 0.05) ** 2) + 0.25 * np.exp(-0.5 * ((x - 0.5) / 0.3) ** 2)

kf = lambda A, B: 0.3 * np.exp(-0.5 * ((A[:, None] - B[None, :]) / 0.06) ** 2)
best_ei, best_rand, true_best = [], [], 1.0
for rep in range(20):
    pref = gb.uniform(0.1, 0.9)
    Xs = [0.1, 0.5, 0.9]; ys = [response(x, pref) + 0.1 * gb.normal() for x in Xs]
    for trial in range(22):
        Xa, ya = np.array(Xs), np.array(ys)
        Kn = kf(Xa, Xa) + 0.01 * np.eye(len(Xa))
        A = np.linalg.solve(Kn, ya)
        mu_g = kf(grid, Xa) @ A
        v_g = np.maximum(0.3 - np.einsum("ij,jk,ik->i", kf(grid, Xa), np.linalg.inv(Kn), kf(grid, Xa)), 1e-9)
        sd_g = np.sqrt(v_g)
        z = (mu_g - max(ya)) / sd_g
        ei = ...
        x_next = grid[int(np.argmax(ei))]
        Xs.append(x_next); ys.append(response(x_next, pref) + 0.1 * gb.normal())
    best_ei.append(max(response(x, pref) for x in Xs))
    rand_x = gb.uniform(0, 1, 25)
    best_rand.append(max(response(x, pref) for x in rand_x))
print(f"best true response found in 25 trials: Bayesian optimisation {np.mean(best_ei):.3f}, random {np.mean(best_rand):.3f} (maximum {true_best:.2f})")

In [ ]:
import numpy as np
def test_finds():
    "Bayesian optimisation gets within 0.05 of the best response in 25 trials, on average over 20 neurons"
    assert np.mean(best_ei) > true_best - 0.05
def test_beats_random():
    "it beats random stimulus selection with the same number of trials"
    assert np.mean(best_ei) > np.mean(best_rand) + 0.02

test_finds()
test_beats_random()
print("All checks passed")